In [1]:
from pathlib import Path
import subprocess
import sys
import numpy as np


# ============================================================
# Repository and dataset location
# ============================================================

REPO_NAME = "3MP-Multimodal-Diagnostic-Platform"
REPO_URL = (
    "https://github.com/GanghaoLiang/"
    "3MP-Multimodal-Diagnostic-Platform.git"
)


def find_repo_root(start_path):
    """
    Search the current directory and its parent directories
    for the repository containing Skin_Cancer_Dataset.
    """
    start_path = Path(start_path).resolve()

    for candidate in [start_path, *start_path.parents]:
        if (candidate / "Skin_Cancer_Dataset").is_dir():
            return candidate

    return None


# First try to locate an already cloned repository.
repo_root = find_repo_root(Path.cwd())

# If running in Google Colab directly from GitHub,
# clone the repository automatically when necessary.
if repo_root is None and "google.colab" in sys.modules:

    clone_dir = Path("/content") / REPO_NAME

    if not clone_dir.exists():
        print("Repository not found locally. Cloning from GitHub...")

        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                REPO_URL,
                str(clone_dir),
            ],
            check=True,
        )

    repo_root = find_repo_root(clone_dir)


if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the repository containing "
        "'Skin_Cancer_Dataset'. "
        "Please run this notebook from within the cloned "
        "3MP-Multimodal-Diagnostic-Platform repository."
    )


dataset_dir = repo_root / "Skin_Cancer_Dataset"

print(f"Repository root: {repo_root}")
print(f"Dataset directory: {dataset_dir}")


# ============================================================
# Load paired biomechanical and biochemical tensors
# ============================================================

def load_and_stack_csv_pair(mech_csv_path, bio_csv_path):
    """
    Load one biomechanical tensor and its paired biochemical
    tensor and combine them into a 96 x 96 x 2 array.
    """

    array_mech = np.loadtxt(
        mech_csv_path,
        delimiter=",",
        dtype=np.float32,
    )

    array_bio = np.loadtxt(
        bio_csv_path,
        delimiter=",",
        dtype=np.float32,
    )

    if array_mech.shape != (96, 96):
        raise ValueError(
            f"Unexpected mechanical tensor shape "
            f"{array_mech.shape} in {mech_csv_path}"
        )

    if array_bio.shape != (96, 96):
        raise ValueError(
            f"Unexpected biochemical tensor shape "
            f"{array_bio.shape} in {bio_csv_path}"
        )

    if not np.all(np.isfinite(array_mech)):
        raise ValueError(
            f"Non-finite values detected in {mech_csv_path}"
        )

    if not np.all(np.isfinite(array_bio)):
        raise ValueError(
            f"Non-finite values detected in {bio_csv_path}"
        )

    array_mech = np.expand_dims(array_mech, axis=-1)
    array_bio = np.expand_dims(array_bio, axis=-1)

    return np.concatenate(
        [array_mech, array_bio],
        axis=-1,
    )


# ============================================================
# Class definitions
# ============================================================

categories = {
    "Normal": 0,
    "Scar": 1,
    "Inflamed": 2,
    "Tumor": 3,
}

expected_counts = {
    "Normal": 74,
    "Scar": 75,
    "Inflamed": 72,
    "Tumor": 163,
}


# ============================================================
# Build the complete dataset
# ============================================================

X_data = []
Y_labels = []
sample_ids = []

print("\nStarting data loading...\n")


for category_name, label_value in categories.items():

    folder_path = dataset_dir / category_name

    if not folder_path.is_dir():
        raise FileNotFoundError(
            f"Missing category directory: {folder_path}"
        )

    # This intentionally searches for "_mech_" rather than
    # a fixed filename suffix.
    #
    # Therefore both of the following are supported:
    #   *_mech_values_values.csv
    #   *_mech_values_values_values.csv
    #
    # This is needed because the Tumor files contain one
    # additional "_values" in their filenames.
    mech_files = sorted(
        [
            file_path
            for file_path in folder_path.iterdir()
            if file_path.is_file()
            and file_path.suffix.lower() == ".csv"
            and "_mech_" in file_path.name
        ]
    )

    loaded_count = 0

    for mech_path in mech_files:

        # Replace only the modality identifier.
        # All remaining filename components are preserved.
        bio_filename = mech_path.name.replace(
            "_mech_",
            "_bio_",
            1,
        )

        bio_path = folder_path / bio_filename

        if not bio_path.is_file():
            raise FileNotFoundError(
                f"Missing biochemical pair for:\n"
                f"  Mechanical: {mech_path.name}\n"
                f"  Expected biochemical: {bio_filename}"
            )

        stacked_tensor = load_and_stack_csv_pair(
            mech_path,
            bio_path,
        )

        X_data.append(stacked_tensor)
        Y_labels.append(label_value)

        # Keep the identifier for traceability.
        sample_ids.append(
            mech_path.name.split("_mech_")[0]
        )

        loaded_count += 1

    print(
        f"Loaded {loaded_count:3d} samples "
        f"from {category_name}."
    )

    if loaded_count != expected_counts[category_name]:
        raise ValueError(
            f"{category_name}: expected "
            f"{expected_counts[category_name]} samples, "
            f"but loaded {loaded_count}."
        )


# ============================================================
# Convert to NumPy arrays and perform integrity checks
# ============================================================

X_data = np.asarray(
    X_data,
    dtype=np.float32,
)

Y_labels = np.asarray(
    Y_labels,
    dtype=np.int32,
)

sample_ids = np.asarray(sample_ids)


print("\nFinal dataset:")
print(f"X_data shape:      {X_data.shape}")
print(f"Y_labels shape:    {Y_labels.shape}")
print(f"Sample IDs shape:  {sample_ids.shape}")

class_counts = np.bincount(
    Y_labels,
    minlength=4,
)

print(f"Class counts:      {class_counts}")


# Final integrity checks
assert X_data.shape == (384, 96, 96, 2), (
    f"Unexpected X_data shape: {X_data.shape}"
)

assert Y_labels.shape == (384,), (
    f"Unexpected Y_labels shape: {Y_labels.shape}"
)

assert sample_ids.shape == (384,), (
    f"Unexpected sample_ids shape: {sample_ids.shape}"
)

assert np.array_equal(
    class_counts,
    np.array([74, 75, 72, 163]),
), f"Unexpected class counts: {class_counts}"


print("\nDataset integrity check PASSED.")

Repository not found locally. Cloning from GitHub...
Repository root: /content/3MP-Multimodal-Diagnostic-Platform
Dataset directory: /content/3MP-Multimodal-Diagnostic-Platform/Skin_Cancer_Dataset

Starting data loading...

Loaded  74 samples from Normal.
Loaded  75 samples from Scar.
Loaded  72 samples from Inflamed.
Loaded 163 samples from Tumor.

Final dataset:
X_data shape:      (384, 96, 96, 2)
Y_labels shape:    (384,)
Sample IDs shape:  (384,)
Class counts:      [ 74  75  72 163]

Dataset integrity check PASSED.


In [2]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks


# ============================================================
# Reproducibility
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Enable deterministic TensorFlow operations when supported.
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


# ============================================================
# Channel-spatial attention block
# ============================================================

def attention_block(inputs):
    channels = inputs.shape[-1]

    # Channel attention
    x = layers.GlobalAveragePooling2D()(inputs)
    x = layers.Dense(
        channels // 4,
        activation="relu"
    )(x)
    x = layers.Dense(
        channels,
        activation="sigmoid"
    )(x)
    x = layers.Reshape(
        (1, 1, channels)
    )(x)

    channel_refined = layers.Multiply()(
        [inputs, x]
    )

    # Spatial attention
    y = layers.Conv2D(
        1,
        (3, 3),
        padding="same",
        activation="sigmoid"
    )(channel_refined)

    spatial_refined = layers.Multiply()(
        [channel_refined, y]
    )

    return spatial_refined


# ============================================================
# CNN model
# ============================================================

def build_attention_cnn(input_shape):

    inputs = layers.Input(
        shape=input_shape
    )

    # Spatial augmentation using horizontal
    # and vertical reflections.
    x = layers.RandomFlip(
        "horizontal_and_vertical",
        seed=SEED
    )(inputs)

    # --------------------------------------------------------
    # Convolution block 1
    # --------------------------------------------------------

    x = layers.Conv2D(
        32,
        (3, 3),
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.MaxPooling2D(
        (2, 2)
    )(x)

    # --------------------------------------------------------
    # Convolution block 2
    # --------------------------------------------------------

    x = layers.Conv2D(
        64,
        (3, 3),
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Dropout(
        0.2,
        seed=SEED
    )(x)

    x = layers.MaxPooling2D(
        (2, 2)
    )(x)

    # --------------------------------------------------------
    # Convolution block 3
    # --------------------------------------------------------

    x = layers.Conv2D(
        128,
        (3, 3),
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    # Channel-spatial attention
    x = attention_block(x)

    x = layers.MaxPooling2D(
        (2, 2)
    )(x)

    # --------------------------------------------------------
    # Deep feature representation
    # --------------------------------------------------------

    x = layers.Flatten()(x)

    x = layers.Dense(
        64,
        use_bias=False
    )(x)

    x = layers.BatchNormalization(
        name="deep_features"
    )(x)

    x = layers.Activation(
        "relu"
    )(x)

    x = layers.Dropout(
        0.5,
        seed=SEED
    )(x)

    # --------------------------------------------------------
    # Four-class classifier
    # --------------------------------------------------------

    outputs = layers.Dense(
        4,
        activation="softmax"
    )(x)

    model = models.Model(
        inputs=inputs,
        outputs=outputs
    )

    # Mild label smoothing for regularization.
    loss_fn = tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=0.05
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=0.001
    )

    model.compile(
        optimizer=optimizer,
        loss=loss_fn,
        metrics=["accuracy"]
    )

    return model


# ============================================================
# TensorFlow training dataset
# ============================================================

def create_standard_dataset(
    X,
    Y,
    batch_size=16
):

    Y_one_hot = tf.one_hot(
        Y,
        depth=4
    )

    ds = tf.data.Dataset.from_tensor_slices(
        (X, Y_one_hot)
    )

    ds = ds.shuffle(
        buffer_size=1024,
        seed=SEED,
        reshuffle_each_iteration=True
    )

    ds = ds.batch(
        batch_size
    )

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )

    return ds


# ============================================================
# Test-time augmentation (TTA)
# ============================================================

def predict_with_tta(
    model,
    X_test
):

    predictions = [

        # Original orientation
        model.predict(
            X_test,
            verbose=0
        ),

        # Horizontal reflection
        model.predict(
            tf.image.flip_left_right(X_test),
            verbose=0
        ),

        # Vertical reflection
        model.predict(
            tf.image.flip_up_down(X_test),
            verbose=0
        ),

        # 90-degree rotation
        model.predict(
            tf.image.rot90(
                X_test,
                k=1
            ),
            verbose=0
        ),

        # 270-degree rotation
        model.predict(
            tf.image.rot90(
                X_test,
                k=3
            ),
            verbose=0
        )
    ]

    # Average class probabilities across
    # the five spatial transformations.
    return np.mean(
        predictions,
        axis=0
    )


print("Cell 2 loaded successfully.")
print(f"TensorFlow version: {tf.__version__}")
print(f"Random seed: {SEED}")

Cell 2 loaded successfully.
TensorFlow version: 2.20.0
Random seed: 42


In [3]:
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold
from tensorflow.keras import callbacks


# ============================================================
# Reproducibility
# ============================================================

# Reset the global random states at the beginning of the
# complete cross-validation experiment.
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)


# ============================================================
# Output directory
# ============================================================

# Keep the published reference results unchanged.
# New runs are written to a separate directory so that they
# can be compared with the originally reported results.
output_dir = repo_root / "Reproduced_Results"
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

print(
    f"Reproduced results will be saved to:\n"
    f"{output_dir}"
)


# ============================================================
# Cross-validation setup
# ============================================================

N_CLASSES = 4
N_SAMPLES = len(Y_labels)

kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)


# Preallocate OOF arrays in the ORIGINAL sample order.
mech_oof_probs = np.zeros(
    (N_SAMPLES, N_CLASSES),
    dtype=np.float32
)

bio_oof_probs = np.zeros(
    (N_SAMPLES, N_CLASSES),
    dtype=np.float32
)

fused_oof_probs = np.zeros(
    (N_SAMPLES, N_CLASSES),
    dtype=np.float32
)

# Record which fold generated each held-out prediction.
oof_fold = np.full(
    N_SAMPLES,
    -1,
    dtype=np.int32
)


# Fold-level accuracies
acc_mech = []
acc_bio = []
acc_fused = []


# ============================================================
# Cosine learning-rate schedule
# ============================================================

def cosine_schedule(epoch, lr):

    total_epochs = 80
    lr_init = 0.001
    lr_min = 0.00001

    return (
        lr_min
        + 0.5
        * (lr_init - lr_min)
        * (
            1
            + np.cos(
                np.pi
                * epoch
                / total_epochs
            )
        )
    )


def make_lr_scheduler():
    """
    Create a fresh learning-rate scheduler for each model.
    """
    return callbacks.LearningRateScheduler(
        cosine_schedule,
        verbose=0
    )


# ============================================================
# Five-fold cross-validation
# ============================================================

print(
    "\nExecuting cosine-annealed "
    "5-fold cross-validation..."
)


for fold_no, (train_index, test_index) in enumerate(
    kfold.split(
        X_data,
        Y_labels
    ),
    start=1
):

    print(
        f"\n========================================"
    )
    print(
        f"Processing Fold {fold_no}/5"
    )
    print(
        f"Training samples: {len(train_index)}"
    )
    print(
        f"Test samples:     {len(test_index)}"
    )
    print(
        f"========================================"
    )

    # --------------------------------------------------------
    # Split the held-out fold
    # --------------------------------------------------------

    X_train_fold = X_data[train_index]
    X_test_fold = X_data[test_index]

    Y_train_fold = Y_labels[train_index]
    Y_test_fold = Y_labels[test_index]


    # --------------------------------------------------------
    # Separate the two modalities
    #
    # Channel 0 = biomechanical
    # Channel 1 = biochemical
    # --------------------------------------------------------

    X_train_mech = X_train_fold[:, :, :, 0:1]
    X_test_mech = X_test_fold[:, :, :, 0:1]

    X_train_bio = X_train_fold[:, :, :, 1:2]
    X_test_bio = X_test_fold[:, :, :, 1:2]


    # ========================================================
    # MODEL 1: BIOMECHANICAL ONLY
    # ========================================================

    print(
        "Training biomechanical-only model..."
    )

    model_mech = build_attention_cnn(
        input_shape=(96, 96, 1)
    )

    model_mech.fit(
        create_standard_dataset(
            X_train_mech,
            Y_train_fold
        ),
        epochs=80,
        callbacks=[
            make_lr_scheduler()
        ],
        verbose=0
    )

    probs_m = predict_with_tta(
        model_mech,
        X_test_mech
    )

    # Place predictions back into their ORIGINAL indices.
    mech_oof_probs[test_index] = probs_m

    fold_acc_mech = np.mean(
        np.argmax(
            probs_m,
            axis=1
        )
        == Y_test_fold
    ) * 100

    acc_mech.append(
        fold_acc_mech
    )

    print(
        f"Biomechanical accuracy: "
        f"{fold_acc_mech:.2f}%"
    )

    del model_mech
    tf.keras.backend.clear_session()


    # ========================================================
    # MODEL 2: BIOCHEMICAL ONLY
    # ========================================================

    print(
        "Training biochemical-only model..."
    )

    model_bio = build_attention_cnn(
        input_shape=(96, 96, 1)
    )

    model_bio.fit(
        create_standard_dataset(
            X_train_bio,
            Y_train_fold
        ),
        epochs=80,
        callbacks=[
            make_lr_scheduler()
        ],
        verbose=0
    )

    probs_b = predict_with_tta(
        model_bio,
        X_test_bio
    )

    bio_oof_probs[test_index] = probs_b

    fold_acc_bio = np.mean(
        np.argmax(
            probs_b,
            axis=1
        )
        == Y_test_fold
    ) * 100

    acc_bio.append(
        fold_acc_bio
    )

    print(
        f"Biochemical accuracy: "
        f"{fold_acc_bio:.2f}%"
    )

    del model_bio
    tf.keras.backend.clear_session()


    # ========================================================
    # MODEL 3: MULTIMODAL FUSION
    # ========================================================

    print(
        "Training multimodal fusion model..."
    )

    model_fused = build_attention_cnn(
        input_shape=(96, 96, 2)
    )

    model_fused.fit(
        create_standard_dataset(
            X_train_fold,
            Y_train_fold
        ),
        epochs=80,
        callbacks=[
            make_lr_scheduler()
        ],
        verbose=0
    )

    probs_f = predict_with_tta(
        model_fused,
        X_test_fold
    )

    fused_oof_probs[test_index] = probs_f

    fold_acc_fused = np.mean(
        np.argmax(
            probs_f,
            axis=1
        )
        == Y_test_fold
    ) * 100

    acc_fused.append(
        fold_acc_fused
    )

    print(
        f"Multimodal accuracy: "
        f"{fold_acc_fused:.2f}%"
    )

    del model_fused
    tf.keras.backend.clear_session()


    # Record the held-out fold for each sample.
    oof_fold[test_index] = fold_no


# ============================================================
# OOF integrity checks
# ============================================================

assert np.all(
    oof_fold > 0
), "Some samples never received an OOF prediction."

assert mech_oof_probs.shape == (
    384,
    4
)

assert bio_oof_probs.shape == (
    384,
    4
)

assert fused_oof_probs.shape == (
    384,
    4
)

# Each probability vector should sum approximately to 1.
assert np.allclose(
    mech_oof_probs.sum(axis=1),
    1.0,
    atol=1e-4
)

assert np.allclose(
    bio_oof_probs.sum(axis=1),
    1.0,
    atol=1e-4
)

assert np.allclose(
    fused_oof_probs.sum(axis=1),
    1.0,
    atol=1e-4
)


# ============================================================
# Pooled OOF accuracy
# ============================================================

mech_pred = np.argmax(
    mech_oof_probs,
    axis=1
)

bio_pred = np.argmax(
    bio_oof_probs,
    axis=1
)

fused_pred = np.argmax(
    fused_oof_probs,
    axis=1
)


pooled_acc_mech = np.mean(
    mech_pred == Y_labels
) * 100

pooled_acc_bio = np.mean(
    bio_pred == Y_labels
) * 100

pooled_acc_fused = np.mean(
    fused_pred == Y_labels
) * 100


print(
    "\n========================================"
)
print(
    "ABLATION RESULTS"
)
print(
    "========================================"
)

print(
    f"Biomechanical-only pooled OOF accuracy: "
    f"{pooled_acc_mech:.2f}%"
)

print(
    f"Biochemical-only pooled OOF accuracy:   "
    f"{pooled_acc_bio:.2f}%"
)

print(
    f"Multimodal fusion pooled OOF accuracy:  "
    f"{pooled_acc_fused:.2f}%"
)


print(
    "\nFold accuracy: mean ± SD"
)

print(
    f"Biomechanical: "
    f"{np.mean(acc_mech):.2f} ± "
    f"{np.std(acc_mech, ddof=1):.2f}%"
)

print(
    f"Biochemical:   "
    f"{np.mean(acc_bio):.2f} ± "
    f"{np.std(acc_bio, ddof=1):.2f}%"
)

print(
    f"Multimodal:    "
    f"{np.mean(acc_fused):.2f} ± "
    f"{np.std(acc_fused, ddof=1):.2f}%"
)

print(
    "========================================"
)


# ============================================================
# Export OOF prediction probabilities
# ============================================================

class_names = [
    "Normal",
    "Scar",
    "Inflamed",
    "Tumor"
]


df_roc = pd.DataFrame(
    {
        "Sample_ID": sample_ids,
        "True_Label": Y_labels,
        "CV_Fold": oof_fold,
    }
)


for i, class_name in enumerate(
    class_names
):

    df_roc[
        f"Mech_Prob_{class_name}"
    ] = mech_oof_probs[:, i]

    df_roc[
        f"Bio_Prob_{class_name}"
    ] = bio_oof_probs[:, i]

    df_roc[
        f"Fused_Prob_{class_name}"
    ] = fused_oof_probs[:, i]


df_roc[
    "Mech_Predicted_Label"
] = mech_pred

df_roc[
    "Bio_Predicted_Label"
] = bio_pred

df_roc[
    "Fused_Predicted_Label"
] = fused_pred


output_file = (
    output_dir
    / "Ablation_ROC_Probabilities.csv"
)

df_roc.to_csv(
    output_file,
    index=False
)


print(
    "\nPipeline complete."
)

print(
    f"OOF probabilities saved to:\n"
    f"{output_file}"
)

Reproduced results will be saved to:
/content/3MP-Multimodal-Diagnostic-Platform/Reproduced_Results

Executing cosine-annealed 5-fold cross-validation...

Processing Fold 1/5
Training samples: 307
Test samples:     77
Training biomechanical-only model...
Biomechanical accuracy: 74.03%
Training biochemical-only model...
Biochemical accuracy: 61.04%
Training multimodal fusion model...
Multimodal accuracy: 90.91%

Processing Fold 2/5
Training samples: 307
Test samples:     77
Training biomechanical-only model...
Biomechanical accuracy: 72.73%
Training biochemical-only model...
Biochemical accuracy: 64.94%
Training multimodal fusion model...
Multimodal accuracy: 87.01%

Processing Fold 3/5
Training samples: 307
Test samples:     77
Training biomechanical-only model...
Biomechanical accuracy: 59.74%
Training biochemical-only model...
Biochemical accuracy: 74.03%
Training multimodal fusion model...
Multimodal accuracy: 88.31%

Processing Fold 4/5
Training samples: 307
Test samples:     77
Tr